# Module 1 — Collecte et compréhension des données
### Projet : House Prices — Advanced Regression Techniques

**Étape du projet :** Data Understanding  
**Source des données :** compétition *House Prices – Advanced Regression Techniques* de Kaggle  
**Couche de données :** Bronze (données brutes)

---

## 1. Présentation du module

Ce module a pour objectif de charger les fichiers mis à disposition par Kaggle et d'effectuer une première exploration de leur structure.

L'analyse porte sur trois fichiers :
- `train.csv` : données d'entraînement, contenant les variables explicatives et la variable cible ;
- `test.csv` : données de test, utilisées pour produire les prédictions ;
- `sample_submission.csv` : exemple du format attendu pour le fichier de soumission.

**Périmètre :** cette étape se limite à la collecte et à la compréhension initiale des données. Le nettoyage, la transformation et la modélisation seront réalisés dans les modules suivants.

## 2. Préparation de l'environnement

Importation des bibliothèques nécessaires et définition du chemin d'accès aux fichiers.

> **Organisation attendue du projet :** les fichiers CSV doivent être placés dans le dossier `data/bronze/`. Le chemin est construit de manière relative afin d'éviter une dépendance à un emplacement propre à un ordinateur.

In [20]:
from pathlib import Path

import numpy as np
import pandas as pd

# Recherche du dossier de données depuis le répertoire courant
candidate_paths = [
    Path.cwd() / "data" / "bronze",
    Path.cwd().parent / "data" / "bronze",
]

DATA_DIR = next((path for path in candidate_paths if path.is_dir()), None)

if DATA_DIR is None:
    raise FileNotFoundError(
        "Dossier 'data/bronze' introuvable. "
        "Vérifiez l'organisation du projet et le répertoire de travail."
    )

print(f"Dossier des données : {DATA_DIR.resolve()}")

Dossier des données : C:\Users\hp\Downloads\house-prices-advanced-regression-techniques\data\bronze


## 3. Collecte des données

Les trois fichiers sont chargés dans des DataFrames Pandas. Les données d'origine sont conservées sans modification à ce stade.

In [21]:
files = {
    "train": "train.csv",
    "test": "test.csv",
    "sample_submission": "sample_submission.csv",
}

dataframes = {}

for name, filename in files.items():
    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {file_path}")

    dataframes[name] = pd.read_csv(file_path)

df_train = dataframes["train"]
df_test = dataframes["test"]
df_submission = dataframes["sample_submission"]

print("Chargement des fichiers terminé.")

Chargement des fichiers terminé.


## 4. Aperçu des jeux de données

Cette première consultation permet de visualiser les premières observations de chaque fichier et de vérifier que leur chargement s'est déroulé correctement.

In [22]:
for name, df in dataframes.items():
    print(f"\n{'=' * 70}\nJEU DE DONNÉES : {name.upper()}\n{'=' * 70}")
    display(df.head())


JEU DE DONNÉES : TRAIN


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000



JEU DE DONNÉES : TEST


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition
0,1461,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,AllPub,...,120,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal
1,1462,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal
2,1463,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal
3,1464,60,RL,78.0,9978,Pave,NaN,IR1,Lvl,AllPub,...,0,0,NaN,NaN,NaN,0,6,2010,WD,Normal
4,1465,120,RL,43.0,5005,Pave,NaN,IR1,HLS,AllPub,...,144,0,NaN,NaN,NaN,0,1,2010,WD,Normal



JEU DE DONNÉES : SAMPLE_SUBMISSION


,Id,SalePrice
0,1461,169277.052498
1,1462,187758.393989
2,1463,183583.683570
3,1464,179317.477511
4,1465,150730.079977


## 5. Dimensions des jeux de données

La dimension indique le nombre d'observations (lignes) et le nombre de variables (colonnes) de chaque jeu de données.

In [23]:
dimensions = pd.DataFrame(
    [
        {
            "Jeu de données": name,
            "Nombre de lignes": df.shape[0],
            "Nombre de colonnes": df.shape[1],
        }
        for name, df in dataframes.items()
    ]
)

display(dimensions)

,Jeu de données,Nombre de lignes,Nombre de colonnes
0,train,1460,81
1,test,1459,80
2,sample_submission,1459,2


## 6. Structure et types des variables

La méthode `info()` présente, pour chaque variable, le nombre de valeurs non nulles et son type de données. Cette information aide à repérer les variables nécessitant une attention particulière lors de la préparation.

In [24]:
for name, df in dataframes.items():
    print(f"\n{'=' * 70}\nSTRUCTURE : {name.upper()}\n{'=' * 70}")
    df.info()


STRUCTURE : TRAIN
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   in

## 7. Première vérification de la qualité des données

Cette section présente le nombre et le pourcentage de valeurs manquantes, ainsi que le nombre de lignes dupliquées. Il s'agit d'un diagnostic initial : aucune correction n'est effectuée dans ce module.

In [25]:
def create_quality_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Retourne un résumé des valeurs manquantes par variable."""
    missing_count = df.isna().sum()
    summary = pd.DataFrame({
        "Valeurs manquantes": missing_count,
        "Pourcentage manquant (%)": (missing_count / len(df) * 100).round(2)
            if len(df) > 0 else 0,
        "Type": df.dtypes.astype(str),
    })
    return summary.sort_values(
        by="Valeurs manquantes",
        ascending=False
    )

for name, df in dataframes.items():
    print(f"\nRésumé des valeurs manquantes — {name.upper()}")
    display(create_quality_summary(df))
    print(f"Nombre de lignes dupliquées : {df.duplicated().sum()}")


Résumé des valeurs manquantes — TRAIN


,Valeurs manquantes,Pourcentage manquant (%),Type
PoolQC,1453,99.52,object
MiscFeature,1406,96.30,object
Alley,1369,93.77,object
Fence,1179,80.75,object
MasVnrType,872,59.73,object
...,...,...,...
ExterQual,0,0.00,object
Exterior2nd,0,0.00,object
Exterior1st,0,0.00,object
RoofMatl,0,0.00,object


Nombre de lignes dupliquées : 0

Résumé des valeurs manquantes — TEST


,Valeurs manquantes,Pourcentage manquant (%),Type
PoolQC,1456,99.79,object
MiscFeature,1408,96.50,object
Alley,1352,92.67,object
Fence,1169,80.12,object
MasVnrType,894,61.27,object
...,...,...,...
Electrical,0,0.00,object
1stFlrSF,0,0.00,int64
2ndFlrSF,0,0.00,int64
LowQualFinSF,0,0.00,int64


Nombre de lignes dupliquées : 0

Résumé des valeurs manquantes — SAMPLE_SUBMISSION


,Valeurs manquantes,Pourcentage manquant (%),Type
Id,0,0.0,int64
SalePrice,0,0.0,float64


Nombre de lignes dupliquées : 0


## 8. Vérification de la variable cible

Dans le jeu d'entraînement, `SalePrice` représente le prix de vente du logement. Cette variable constitue la cible à prédire dans le cadre du problème de régression.

Le jeu de test ne contient normalement pas cette variable : il servira à générer les prédictions.

In [26]:
target = "SalePrice"

if target in df_train.columns:
    print("Statistiques descriptives de la variable cible :")
    display(df_train[target].describe().to_frame(name="SalePrice"))
else:
    print(f"La variable cible '{target}' n'a pas été trouvée dans train.csv.")

print(f"Présence de '{target}' dans train.csv : {target in df_train.columns}")
print(f"Présence de '{target}' dans test.csv  : {target in df_test.columns}")

Statistiques descriptives de la variable cible :


,SalePrice
count,1460.000000
mean,180921.195890
std,79442.502883
min,34900.000000
25%,129975.000000
50%,163000.000000
75%,214000.000000
max,755000.000000


Présence de 'SalePrice' dans train.csv : True
Présence de 'SalePrice' dans test.csv  : False


## 9. Synthèse de la compréhension des données

À l'issue de ce module, les fichiers d'entraînement, de test et de soumission ont été chargés et leur structure a été examinée.

Les contrôles réalisés portent sur :
- les dimensions des jeux de données ;
- les types et le nombre de valeurs non nulles des variables ;
- les valeurs manquantes ;
- les lignes dupliquées ;
- l'identification de la variable cible `SalePrice`.

**Conclusion :** les résultats de cette exploration constituent le point de départ de la préparation des données. Les anomalies éventuelles devront être étudiées avant toute transformation ou entraînement de modèle.

**Prochaine étape :** approfondir l'analyse des variables et définir les traitements nécessaires dans le module de préparation des données.